In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Import Model 
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Config 
import torch 
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Model adjustement 
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)


In [ ]:
print(model)

In [ ]:
from peft import LoraConfig, get_peft_model

# Config for lora
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],
    task_type="CAUSAL_LM",
    fan_in_fan_out=True
)

In [ ]:
model = get_peft_model(
    model,
    lora_config
)
model.print_trainable_parameters()

In [1]:
import json

data = [
    {
        "question": "what is python?",
        "answer": "python is a high level programming language."
    },
    {
        "question": "what is machine learning?",
        "answer": "machine learning is a method where computers learn patterns from data."
    },
    {
        "question": "what is AI?",
        "answer": "AI is the field of building systems that can perform tasks requiring intelligence."
    }
]

with open("dataset.json", "w") as file:
    json.dump(data, file, indent=4)


from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="dataset.json"
)

print(dataset)

/Users/debajyotihazra/Documents/Fine Tune/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 3 examples [00:00, 830.34 examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 3
    })
})


In [ ]:
print(dataset["train"][0])

In [ ]:
def format_data(example):
    return {
        "text": f"Question: {example['question']}\n"
                f"Answer: {example['answer']}"
    }

dataset = dataset.map(format_data)
print(dataset["train"][0])

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenize_dataset = dataset.map(tokenize)
print(tokenize_dataset["train"][0])

In [ ]:
def add_labels(example):
    example["labels"] = [
        token if mask == 1 else -100
        for token, mask in zip(
            example["input_ids"],
            example["attention_mask"]
        )
    ]
    return example

tokenize_dataset = tokenize_dataset.map(add_labels)

# Attention use 
print(tokenize_dataset["train"][0]["labels"])


In [ ]:
# Train 
from transformers import Trainer, TrainingArguments

training_arguments = TrainingArguments(
    output_dir="./qlora-output",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    learning_rate=5e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenize_dataset["train"]
)

trainer.train()

In [ ]:
model.save_pretrained("./qlora-adapter")
tokenizer.save_pretrained("./qlora-adapter")

In [ ]:
import os

print(os.listdir("./qlora-adapter"))

In [ ]:
import torch

model.eval()

prompt = "Question: what is python?\nAnswer:"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=30
    )

print(
    tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )
)